# 02 - Preparar dados para analise

Este notebook carrega os Parquets brutos convertidos, seleciona descricoes de produtos, remove nulos e vazios, normaliza texto e consolida duplicatas.

In [2]:
from __future__ import annotations

import sys
import re
import unicodedata
import os
from pathlib import Path

import pandas as pd

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    source_dir = path / "src"
    if source_dir.exists():
        sys.path.append(str(source_dir))
        break
else:
    raise FileNotFoundError("Nao encontrei a pasta src nos diretorios pais.")

from nofis_classifier.paths import find_project_root

## Configuracao

In [3]:
DATASET_NAME = os.getenv("DATASET_NAME", "items-notas-fiscais")
FREQUENCY_COLUMN = "frequencia"

PROJECT_ROOT = find_project_root()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / DATASET_NAME
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME

INTERIM_DIR, PROCESSED_DIR

(WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/interim/items-notas-fiscais'),
 WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/processed/items-notas-fiscais'))

## Funcoes de preparacao

In [13]:
def normalize_column_name(column_name: str) -> str:
    """Normalize a column name enough to detect expected fields."""
    text = unicodedata.normalize("NFKD", str(column_name))
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = text.upper()
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def find_column(columns: pd.Index, required_tokens: tuple[str, ...]) -> str:
    """Find the first column whose normalized name contains all required tokens."""
    for column in columns:
        normalized = normalize_column_name(column)
        if all(token in normalized for token in required_tokens):
            return column
    raise KeyError(f"Nao encontrei coluna com tokens {required_tokens}. Colunas: {list(columns)}")


def normalize_description(value: object) -> str:
    """Normalize product descriptions for TF-IDF and duplicate consolidation."""
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_month(period: str) -> str:
    """Convert YYYY-MM to YYYYMM for artifact names."""
    return period.replace("-", "")


def most_frequent_by_description(
    frame: pd.DataFrame,
    value_column: str,
    output_column: str,
) -> pd.DataFrame:
    """Return the most common value per normalized description."""
    valid = frame[["descricao_normalizada", value_column]].dropna(subset=[value_column]).copy()
    valid = valid[valid[value_column].astype(str).str.strip() != ""]
    if valid.empty:
        return pd.DataFrame(columns=["descricao_normalizada", output_column])

    weighted = (
        valid.groupby(["descricao_normalizada", value_column], as_index=False)
        .size()
        .sort_values(["descricao_normalizada", "size"], ascending=[True, False])
    )
    return weighted.drop_duplicates("descricao_normalizada")[["descricao_normalizada", value_column]].rename(
        columns={value_column: output_column}
    )

## Testes rapidos da normalizacao

In [14]:
normalization_examples = {
    " ARROZ TIPO 1 5KG ": "arroz tipo 1 5kg",
    "CAFÉ TORRADO 500g": "cafe torrado 500g",
    "LEITE UHT 1L": "leite uht 1l",
    "AÇÚCAR-CRISTAL 2 KG": "acucar cristal 2 kg",
}

for raw, expected in normalization_examples.items():
    normalized = normalize_description(raw)
    assert normalized == expected, (raw, normalized, expected)
    assert any(char.isalpha() for char in normalized), (raw, normalized)

duplicate_check = pd.DataFrame(
    {
        "descricao_original": ["Arroz Tipo 1", " arroz tipo 1 ", "ARROZ TIPO 1"],
    }
)
duplicate_check["descricao_normalizada"] = duplicate_check["descricao_original"].map(normalize_description)
assert duplicate_check.groupby("descricao_normalizada").size().iloc[0] == 3

## Carregar Parquets intermediarios

In [15]:
parquet_files = sorted(INTERIM_DIR.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"Nenhum Parquet encontrado em {INTERIM_DIR}")

frames = []
for parquet_file in parquet_files:
    frame = pd.read_parquet(parquet_file)
    frame["competencia"] = parquet_file.stem
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

print(f"Arquivos carregados: {len(parquet_files)}")
print(f"Linhas intermediarias: {len(df):,}")
df.head()

Arquivos carregados: 4
Linhas intermediarias: 2,237,538


,CHAVE DE ACESSO,MODELO,SÉRIE,NÚMERO,NATUREZA DA OPERAÇÃO,DATA EMISSÃO,EVENTO MAIS RECENTE,DATA/HORA EVENTO MAIS RECENTE,CPF/CNPJ Emitente,RAZÃO SOCIAL EMITENTE,...,NÚMERO PRODUTO,DESCRIÇÃO DO PRODUTO/SERVIÇO,CÓDIGO NCM/SH,NCM/SH (TIPO DE PRODUTO),CFOP,QUANTIDADE,UNIDADE,VALOR UNITÁRIO,VALOR TOTAL,competencia
0,32250148658655000188550010000007301109131230,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,730,VENDA MERC DENTRO ESTADO,01/01/2025 00:00:00,Autorização de Uso,01/01/2025 21:16:33,48658655000188,FADINI SOLUCOES LTDA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01
1,51250104695497000160550110000434184166743940,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,11,43418,VENDA DE COMBUSTIVEL OU LUBRIFICANTE ADQUIRIDO...,01/01/2025 06:19:19,Autorização de Uso,01/01/2025 07:04:45,04695497000160,C. MITIKO SACUNO LTDA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01
2,51250104695497000160550110000434191056340194,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,11,43419,VENDA DE COMBUSTIVEL OU LUBRIFICANTE ADQUIRIDO...,01/01/2025 06:26:54,Autorização de Uso,01/01/2025 06:26:56,04695497000160,C. MITIKO SACUNO LTDA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01
3,33250128503456000193550020000440181101927108,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,2,44018,VENDA,01/01/2025 06:41:12,Autorização de Uso,01/01/2025 06:41:15,28503456000193,POSTO DOS PESCADORES LTDA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01
4,33250129051570000192550060000573941110143139,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,6,57394,VENDA,01/01/2025 06:43:55,Autorização de Uso,01/01/2025 06:43:55,29051570000192,POSTO DOS SANTOS REIS LTDA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01


## Selecionar descricoes

In [16]:
description_col = find_column(df.columns, ("DESCRI", "PRODUTO"))

support_columns = {
    "codigo_ncm": None,
    "ncm_tipo": None,
}

try:
    support_columns["codigo_ncm"] = find_column(df.columns, ("NCM", "SH"))
except KeyError:
    pass

try:
    support_columns["ncm_tipo"] = find_column(df.columns, ("TIPO", "PRODUTO"))
except KeyError:
    pass

selected_columns = [description_col, "competencia"]
selected_columns.extend(column for column in support_columns.values() if column is not None)

rename_columns = {description_col: "descricao_original"}
if support_columns["codigo_ncm"] is not None:
    rename_columns[support_columns["codigo_ncm"]] = "codigo_ncm"
if support_columns["ncm_tipo"] is not None:
    rename_columns[support_columns["ncm_tipo"]] = "ncm_tipo"

df_selected = df[selected_columns].rename(columns=rename_columns)

df_selected.head()

,descricao_original,competencia,codigo_ncm,ncm_tipo
0,NaN,2025-01,NaN,NaN
1,NaN,2025-01,NaN,NaN
2,NaN,2025-01,NaN,NaN
3,NaN,2025-01,NaN,NaN
4,NaN,2025-01,NaN,NaN


## Remover nulos, normalizar e consolidar duplicatas

In [17]:
input_rows = len(df_selected)

df_valid = df_selected.dropna(subset=["descricao_original"]).copy()
df_valid["descricao_original"] = df_valid["descricao_original"].astype(str).str.strip()
df_valid = df_valid[df_valid["descricao_original"] != ""].copy()

df_valid["descricao_normalizada"] = df_valid["descricao_original"].map(normalize_description)
df_valid = df_valid[df_valid["descricao_normalizada"] != ""].copy()

valid_rows = len(df_valid)

print(f"Linhas com coluna de descricao: {input_rows:,}")
print(f"Linhas com descricao valida: {valid_rows:,}")
print(f"Linhas removidas por nulos/vazios: {input_rows - valid_rows:,}")

Linhas com coluna de descricao: 2,237,538
Linhas com descricao valida: 1,764,795
Linhas removidas por nulos/vazios: 472,743


In [18]:
aggregations = {
    "frequencia": ("descricao_original", "size"),
    "descricao_original_exemplo": ("descricao_original", "first"),
    "descricoes_originais_distintas": ("descricao_original", "nunique"),
    "primeiro_mes": ("competencia", "min"),
    "ultimo_mes": ("competencia", "max"),
    "meses_presentes": ("competencia", "nunique"),
}

df_modeling = (
    df_valid.groupby("descricao_normalizada", as_index=False)
    .agg(**aggregations)
    .sort_values(["frequencia", "descricao_normalizada"], ascending=[False, True])
    .reset_index(drop=True)
)

if "codigo_ncm" in df_valid.columns:
    ncm_mode = most_frequent_by_description(df_valid, "codigo_ncm", "codigo_ncm_mais_frequente")
    df_modeling = df_modeling.merge(ncm_mode, on="descricao_normalizada", how="left")

if "ncm_tipo" in df_valid.columns:
    ncm_tipo_mode = most_frequent_by_description(df_valid, "ncm_tipo", "ncm_tipo_mais_frequente")
    df_modeling = df_modeling.merge(ncm_tipo_mode, on="descricao_normalizada", how="left")

df_modeling.head(10)

,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,personalizacao novo passaporte brasileiro paco...,15481,PERSONALIZACAO NOVO PASSAPORTE BRASILEIRO PACO...,1,2025-01,2025-04,4,49119900.0,Outros impressos
1,cenoura,5435,CENOURA,5,2025-01,2025-04,4,7061000.0,"Cenouras e nabos, frescos ou refrigerados"
2,tomate,4450,TOMATE,7,2025-01,2025-04,4,7020000.0,"Tomates, frescos ou refrigerados"
3,beterraba,4447,BETERRABA,6,2025-01,2025-04,4,7069000.0,"Beterrabas, rabanetes e outras raízes, frescas..."
4,melancia,4409,MELANCIA,7,2025-01,2025-04,4,8071100.0,Melancias frescas
5,banana prata,4116,Banana prata,12,2025-01,2025-04,4,8039000.0,"Bananas frescas ou secas, exceto bananas-da-terra"
6,cebola,3996,CEBOLA,4,2025-01,2025-04,4,7122000.0,"Cebolas secas, inclusive pedaços, fatias, pó, ..."
7,gasolina comum,3986,GASOLINA COMUM,4,2025-01,2025-04,4,27101259.0,"Outras gasolinas, exceto para aviação"
8,batata doce,3967,BATATA DOCE,11,2025-01,2025-04,4,7142000.0,"Batatas-doces, frescas, refrigeradas, congelad..."
9,abobora,3914,ABOBORA,9,2025-01,2025-04,4,7099300.0,"Abóboras, abobrinhas e cabaças, fresca ou refr..."


## Validacoes finais

In [19]:
assert not df_modeling.empty
assert df_modeling["descricao_normalizada"].notna().all()
assert (df_modeling["descricao_normalizada"].str.strip() != "").all()
assert int(df_modeling["frequencia"].sum()) == valid_rows
assert df_modeling["descricao_normalizada"].is_unique

print(f"Descricoes unicas normalizadas: {len(df_modeling):,}")
print(f"Linhas validas consolidadas: {int(df_modeling['frequencia'].sum()):,}")

Descricoes unicas normalizadas: 466,284
Linhas validas consolidadas: 1,764,795


## Salvar base processada

In [20]:
start_period = compact_month(min(file.stem for file in parquet_files))
end_period = compact_month(max(file.stem for file in parquet_files))
output_path = PROCESSED_DIR / f"{DATASET_NAME}-{start_period}-{end_period}.parquet"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
if output_path.exists():
    raise FileExistsError(
        f"Arquivo processado ja existe: {output_path}. "
        "Escolha outro nome ou remova o arquivo manualmente se quiser regenerar."
    )

df_modeling.to_parquet(output_path, index=False)

output_path

WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/processed/items-notas-fiscais/items-notas-fiscais-202501-202504.parquet')